# 02_b_premodel_steps: Marketing-Aware Pre-Model Pipeline

This notebook converts outputs from 01_b, 01_c, and 02_a into a model-ready pipeline with explicit strategy choices.

Objectives:
1. Build leakage-safe train/validation split.
2. Prepare features with business constraints from EDA findings.
3. Train baseline models with consistent preprocessing.
4. Compare model quality using classification + ranking metrics.
5. Select decision threshold aligned to campaign objective.
6. Export prepared datasets, feature lists, and model artifacts.

## Step 1) Imports and Data Loading

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    f1_score, precision_score, recall_score, roc_auc_score, average_precision_score,
    mean_squared_error
    )
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

pd.set_option('display.max_columns', 200)

PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
INPUT_PATH = DATA_DIR / 'features_added' / 'preprocessed_data_features_added.csv'

if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Expected file not found: {INPUT_PATH}')

df = pd.read_csv(INPUT_PATH)
print('Loaded:', INPUT_PATH)
print('Shape:', df.shape)
df.head()

Loaded: d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_data_driven_marketing\starbucks_reward_data_marketing\data\features_added\preprocessed_data_features_added.csv
Shape: (50637, 50)


,person,offer_id,time_completed,time_received,time_viewed,was_viewed,completed_after_view,within_offer_window,offer_success,received_not_viewed,age,income,days_since_registration,age_was_118,income_missing_before_impute,gender_missing_before_fill,age_missing_before_impute,reward,difficulty,channel_email,channel_mobile,channel_social,channel_web,channel_count,reward_to_difficulty,reward_per_day,offer_type_bogo,offer_type_discount,offer_type_informational,transaction_count,total_spent,avg_transaction_value,duration,time_completed_was_imputed,time_viewed_was_imputed,age_group,income_group,membership_duration_months,has_web_channel,has_email_channel,has_mobile_channel,has_social_channel,spend_per_transaction,spend_per_membership_day,difficulty_per_day,net_value_score,gender_F,gender_M,gender_O,gender_U
0,0009655768c64bdeb2e877511632db8f,2906b810c7d4411798c6938adc9daaa5,576.0,576.0,360.0,0,False,True,0,1,33,72000.0,461,0,0,0,0,2,10,1,1,0,1,3,0.20,0.285714,0,1,0,8.0,127.60,15.950000,7,0,1,26-35,middle,15.14,1,1,1,0,15.950000,0.276790,1.428571,1.0,False,True,False,False
1,0009655768c64bdeb2e877511632db8f,f19421c1d4aa40978ebb69ca19b0e20d,414.0,408.0,456.0,1,False,True,0,0,33,72000.0,461,0,0,0,0,5,5,1,1,1,1,4,1.00,1.000000,1,0,0,8.0,127.60,15.950000,5,0,0,26-35,middle,15.14,1,1,1,1,15.950000,0.276790,1.000000,4.5,False,True,False,False
2,0009655768c64bdeb2e877511632db8f,fafdcd668e3743c1bb461111dcafc2a4,528.0,504.0,540.0,1,False,True,0,0,33,72000.0,461,0,0,0,0,2,10,1,1,1,1,4,0.20,0.200000,0,1,0,8.0,127.60,15.950000,10,0,0,26-35,middle,15.14,1,1,1,1,15.950000,0.276790,1.000000,1.0,False,True,False,False
3,00116118485d4dfda04fdbaba9a87b5c,f19421c1d4aa40978ebb69ca19b0e20d,420.0,168.0,216.0,1,False,False,0,0,55,64000.0,92,1,1,1,1,5,5,1,1,1,1,4,1.00,1.000000,1,0,0,3.0,4.09,1.363333,5,1,0,46-55,middle,3.02,1,1,1,1,1.363333,0.044457,1.000000,4.5,False,False,False,True
4,0011e0d4e6b944f998e987f904e8c1e5,0b1e1539f2cc45b7b9fa7c272da2e1d7,576.0,408.0,432.0,1,True,True,1,0,40,57000.0,198,0,0,0,0,5,20,1,0,0,1,2,0.25,0.500000,0,1,0,5.0,79.46,15.892000,10,0,0,36-45,lower_middle,6.50,1,1,0,0,15.892000,0.401313,2.000000,3.0,False,False,True,False


## Step 2) Target Definition and Governance Rules

Rules from EDA and business strategy:
- Target: `offer_success` (fallback to compatible success columns if needed).
- Remove obvious identifiers from features.
- Remove leakage-prone columns tied directly to completion outcomes.

In [2]:
if 'offer_success' in df.columns:
    target_col = 'offer_success'
elif 'offer_completed' in df.columns:
    target_col = 'offer_completed'
elif 'offer_completed_event' in df.columns:
    target_col = 'offer_completed_event'
else:
    raise ValueError('No target found: expected offer_success / offer_completed / offer_completed_event.')

df[target_col] = pd.to_numeric(df[target_col], errors='coerce').fillna(0).astype(int)

id_like_cols = [c for c in ['person', 'offer_id', 'offer'] if c in df.columns]
leakage_cols = [c for c in ['completed_after_view', 'within_offer_window', 'time_completed', 'time_completed_was_imputed'] if c in df.columns]

drop_feature_cols = set(id_like_cols + leakage_cols + [target_col])
feature_cols = [c for c in df.columns if c not in drop_feature_cols]

X = df[feature_cols].copy()
y = df[target_col].copy()

print('Target:', target_col)
print('Target balance (%):')
print((y.value_counts(normalize=True).sort_index() * 100).round(2))
print('ID-like dropped:', id_like_cols)
print('Leakage dropped:', leakage_cols)
print('Feature count:', len(feature_cols))

Target: offer_success
Target balance (%):
offer_success
0    61.02
1    38.98
Name: proportion, dtype: float64
ID-like dropped: ['person', 'offer_id']
Leakage dropped: ['completed_after_view', 'within_offer_window', 'time_completed', 'time_completed_was_imputed']
Feature count: 43


## Step 3) Per-User Leave-Last Temporal Split

Use each customer's earlier offers for training and reserve their last chronological offer for validation.
This matches the marketing problem better than a random group split because it preserves user history and avoids the 100% cold-start issue seen in collaborative filtering.

In [26]:
if 'person' not in df.columns:
    raise ValueError('Expected a person column for per-user temporal splitting.')
if 'time_received' not in df.columns:
    raise ValueError('Expected a time_received column for per-user temporal splitting.')

split_view = df[['person', 'time_received']].copy()
split_view['_row_order'] = np.arange(len(df))
split_view['_time_received'] = pd.to_numeric(split_view['time_received'], errors='coerce')

if split_view['_time_received'].isna().any():
    raise ValueError('time_received contains missing or non-numeric values, so the temporal split cannot be trusted.')

split_view = split_view.sort_values(['person', '_time_received', '_row_order'], kind='mergesort')
train_rows = []
valid_rows = []
single_interaction_users = 0

for _, group in split_view.groupby('person', sort=False):
    if len(group) == 1:
        train_rows.extend(group['_row_order'].tolist())
        single_interaction_users += 1
    else:
        train_rows.extend(group.iloc[:-1]['_row_order'].tolist())
        valid_rows.append(int(group.iloc[-1]['_row_order']))

train_idx = np.array(train_rows, dtype=int)
valid_idx = np.array(valid_rows, dtype=int)

X_train, X_valid = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
y_train, y_valid = y.iloc[train_idx].copy(), y.iloc[valid_idx].copy()

train_users = set(df.iloc[train_idx]['person'])
valid_users = set(df.iloc[valid_idx]['person'])

print('Split strategy: per-user leave-last temporal split')
print('Train shape:', X_train.shape, '| Valid shape:', X_valid.shape)
print('Train target rate:', round(y_train.mean(), 4))
print('Valid target rate:', round(y_valid.mean(), 4))

print('Users in validation:', len(valid_users))
print('Validation users covered by training history:', len(train_users & valid_users))
print('Users with only one interaction kept in train:', single_interaction_users)

Split strategy: per-user leave-last temporal split
Train shape: (34747, 43) | Valid shape: (15890, 43)
Train target rate: 0.395
Valid target rate: 0.3786
Users in validation: 15890
Validation users covered by training history: 15890
Users with only one interaction kept in train: 1038


## Step 4) Preprocessing Pipeline: Feature Engineering & Normalization Strategy

**Why this step matters:**
Preprocessing transforms raw features into a consistent, model-ready format that allows baseline classifiers to train effectively and fairly. This step is critical because:
- **Numerical stability**: Unscaled numeric features with vastly different ranges (e.g., age vs. spend amount) can cause gradient-based models to converge slowly or toward suboptimal solutions.
- **Categorical interpretability**: One-hot encoding converts categorical features (offer_type, channel) into a format that tree-based and linear models can process while preserving interpretability.
- **Reproducibility**: A single preprocessor object ensures that train, validation, and future production data follow the identical transformation rules, preventing data leakage and distribution shift.

### Strategic Approach:

**For Numeric Features:**
- **Median Imputation**: Robust to outliers compared to mean; preserves the 50th percentile of each feature's distribution.
- **Standard Scaling**: Centers and scales each feature to mean ~0, std ~1. Critical for Logistic Regression (linear model), and beneficial for tree models (improves feature comparison logic).
- **Boolean → Integer**: Converts boolean columns to 0/1 numeric values so they flow through numeric pipeline instead of being treated as categorical strings.

**For Categorical Features:**
- **Most-Frequent Imputation**: Forward-fills missing values with the dataset mode, preserving the dominant category without introducing artificial variance.
- **One-Hot Encoding** (`handle_unknown='ignore'`): 
  - Creates binary columns for each category (e.g., `offer_type=discount` → `offer_type_discount=1`).
  - Handles unseen categories in validation/production gracefully by setting all one-hot columns to 0 (safe fallback).
  - Enables linear models to learn category-specific coefficients and tree models to make splits on category presence.

### Feature Engineering Overview:
- **Offer Features**: One-hot encode `offer_type` (discount, BOGO, informational) to capture offer strategy nuances.
- **Channel Features**: One-hot encode `channel` (web, email, mobile, social) to reflect where customers receive and respond to offers.
- **Customer Demographics**: Numeric age, income; categorical age_group, income_group to capture both fine-grained and segment-level patterns.
- **Behavioral Features**: Scaled spend metrics, tenure, transaction counts to reflect customer engagement and value.

### Single Reusable Transformer:
One `ColumnTransformer` object is fit on the training set and applied to both validation and all baseline models. This ensures:
- **Consistency**: All models receive identically transformed inputs.
- **No data leakage**: Scaler statistics (mean, std, vocabulary) come only from training data.
- **Portability**: The same fitted transformer can be serialized and used in production inference.

In [27]:
# Convert boolean cols → int để chúng được xử lý như numeric
bool_cols = X_train.select_dtypes(include=['bool']).columns.tolist()
X_train[bool_cols] = X_train[bool_cols].astype(int)
X_valid[bool_cols] = X_valid[bool_cols].astype(int)

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols)
], remainder='drop')

print('Numeric cols:', len(numeric_cols))
print('Categorical cols:', len(categorical_cols))

Numeric cols: 41
Categorical cols: 2


## Step 4b) Most Popular Offer Baseline (Naive Benchmark)

Không dùng model. Predict = tỷ lệ success trung bình theo từng offer trên train set.
Dùng để chứng minh supervised models có giá trị gia tăng.


In [28]:
# ── Most Popular Offer Baseline ─────────────────────────────────
train_offer_ids = df.iloc[train_idx]['offer_id']
valid_offer_ids = df.iloc[valid_idx]['offer_id']

# Completion rate trung bình theo offer trên train
offer_pop_rate = (
    pd.DataFrame({'offer_id': train_offer_ids, 'y': y_train})
    .groupby('offer_id')['y'].mean()
)

# Map sang valid set
valid_proba_popular = valid_offer_ids.map(offer_pop_rate).fillna(y_train.mean()).values
valid_pred_popular  = (valid_proba_popular >= 0.5).astype(int)

popular_metrics = {
    'model': 'most_popular_offer',
    'accuracy': accuracy_score(y_valid, valid_pred_popular),
    'balanced_accuracy': balanced_accuracy_score(y_valid, valid_pred_popular),
    'f1': f1_score(y_valid, valid_pred_popular),
    'precision': precision_score(y_valid, valid_pred_popular, zero_division=0),
    'recall': recall_score(y_valid, valid_pred_popular, zero_division=0),
    'roc_auc': roc_auc_score(y_valid, valid_proba_popular),
    'pr_auc': average_precision_score(y_valid, valid_proba_popular),
}

print("Most Popular Offer Baseline:")
for k, v in popular_metrics.items():
    if k != 'model':
        print(f"  {k}: {v:.4f}")

print("\nOffer-level popularity (train):")
print(offer_pop_rate.sort_values(ascending=False).to_string())


Most Popular Offer Baseline:
  accuracy: 0.6581
  balanced_accuracy: 0.6048
  f1: 0.4603
  precision: 0.5720
  recall: 0.3851
  roc_auc: 0.6683
  pr_auc: 0.5100

Offer-level popularity (train):
offer_id
fafdcd668e3743c1bb461111dcafc2a4    0.645528
2298d6c36e964ae4a3e7e9706d1fb8c2    0.585678
f19421c1d4aa40978ebb69ca19b0e20d    0.463953
4d5c57ea9a6940dd891ad53e9dbe8da0    0.361092
ae264e3637204a6fb9bb56bc8210ddfd    0.344422
2906b810c7d4411798c6938adc9daaa5    0.291311
9b98b8c7a33c4b65b9aebfe6a799e6d9    0.282508
0b1e1539f2cc45b7b9fa7c272da2e1d7    0.193094


## Step 5) Baseline Model Training and Evaluation

We train three strong baselines and compare both classification and ranking quality:
- Logistic Regression
- Random Forest
- Histogram Gradient Boosting

In [29]:
def fit_and_evaluate(model_name, estimator):
    pipe = Pipeline([
        ('prep', preprocessor),
        ('model', estimator)
    ])
    pipe.fit(X_train, y_train)

    pred = pipe.predict(X_valid)
    if hasattr(pipe, 'predict_proba'):
        proba = pipe.predict_proba(X_valid)[:, 1]
    else:
        decision = pipe.decision_function(X_valid)
        proba = (decision - decision.min()) / (decision.max() - decision.min() + 1e-9)

    metrics = {
        'model': model_name,
        'accuracy': accuracy_score(y_valid, pred),
        'balanced_accuracy': balanced_accuracy_score(y_valid, pred),
        'f1': f1_score(y_valid, pred),
        'precision': precision_score(y_valid, pred, zero_division=0),
        'recall': recall_score(y_valid, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_valid, proba),
        'pr_auc': average_precision_score(y_valid, proba)
    }
    return pipe, metrics, pred, proba

models = {
    'log_reg': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    'random_forest': RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=5,
        class_weight='balanced_subsample', random_state=42, n_jobs=-1
    ),
    'hist_gb': HistGradientBoostingClassifier(
        learning_rate=0.05, max_depth=8, max_iter=300, random_state=42
    )
}

results = []
fitted_models = {}
pred_cache = {}
proba_cache = {}

for name, est in models.items():
    pipe, m, pred, proba = fit_and_evaluate(name, est)
    fitted_models[name] = pipe
    pred_cache[name] = pred
    proba_cache[name] = proba
    results.append(m)

results_df = pd.DataFrame(results).sort_values('pr_auc', ascending=False).reset_index(drop=True)
results_df

,model,accuracy,balanced_accuracy,f1,precision,recall,roc_auc,pr_auc
0,hist_gb,0.829075,0.831617,0.788605,0.741511,0.842088,0.907268,0.821336
1,random_forest,0.800629,0.795869,0.746722,0.719347,0.776263,0.888374,0.788355
2,log_reg,0.789050,0.806393,0.759092,0.668650,0.877826,0.882908,0.769039


In [30]:
# ── MSE cho tất cả supervised models ────────────────────────────
print("\n=== MSE (predicted probability vs actual label) ===")
for name, proba in proba_cache.items():
    mse = mean_squared_error(y_valid, proba)
    print(f"  {name}: MSE = {mse:.4f}")



=== MSE (predicted probability vs actual label) ===
  log_reg: MSE = 0.1442
  random_forest: MSE = 0.1302
  hist_gb: MSE = 0.1190


## Step 5b) SVD Collaborative Filtering: User-Offer Interaction Matrix Decomposition

**Why this approach complements supervised baselines:**
Collaborative filtering discovers hidden patterns in customer-offer interactions without explicit feature engineering. With the leave-last temporal split, every validation row is a next-offer prediction for a user already seen in training, which is the right setting for SVD. It answers: *"Which customers are similar to each other, and which offers appeal to similar customers?"* This is fundamentally different from supervised learning's *"Which features predict offer success?"*

### Strategic Importance:

1. **Uncover Latent Customer Segments**: 
   - SVD decomposes the user-offer matrix into latent factors (low-rank approximation) that represent hidden customer preferences or offer appeal dimensions.
   - Example: Latent factor 1 might represent "budget-conscious" customers who respond to discounts; Latent factor 2 might represent "social-media savvy" customers attracted to mobile/app offers.
   - These factors are learned purely from interaction patterns, without explicit demographic labels.

2. **Handle Sparse Interactions**:
   - Not all customers see all offers. The user-offer matrix is highly sparse (many zeros).
   - SVD + low-rank approximation gracefully handles sparsity by leveraging patterns from similar customers and offers to predict missing interactions.

3. **Cross-Domain Generalization**:
   - Supervised models learned from rich demographic/behavioral features may overfit to training distribution.
   - Collaborative filtering on interaction patterns alone provides an orthogonal signal: it predicts based on peer similarity, not learned coefficients.

### Strategic Implementation:

**Step 1: Build User-Offer Interaction Matrix (Train Set Only)**
- Rows = unique customers (person_id)
- Columns = unique offers (offer_id)
- Values = target (offer_success): 1 if customer responded to offer, 0 otherwise
- Benefit: Captures customer-offer affinity on the training set only (no validation data leakage).

**Step 2: SVD Decomposition to k Latent Factors**
- Singular Value Decomposition: `M = U × Σ × V^T`
  - `U` (m × k): Customer latent representations (how each customer scores on hidden factors)
  - `Σ` (k × k): Strength of each latent factor (relevance ranking)
  - `V^T` (k × n): Offer latent representations (how each offer aligns with hidden factors)
- Reconstruction: `M_reconstructed = U × Σ × V^T` approximates the original matrix with lower rank.
- **k=5 choice**: Balances expressiveness (capture meaningful patterns) vs. noise resistance (avoid overfitting sparse data).

**Step 3: Predict on Validation Set**
- For each validation row (customer, offer):
  - If both customer and offer appear in training matrix → use reconstructed score from SVD
  - If either is unseen → fallback to global training success rate (conservative estimate)
- Clip predictions to [0, 1] to treat them as probabilities.

**Step 4: Benchmark Against Supervised Models**
- Compare PR-AUC, ROC-AUC, and F1 with supervised baselines.
- If SVD performs competitively: indicates that offer success is largely driven by customer-offer fit (collaborative signal) rather than rich features alone.
- If supervised models outperform: indicates that demographic/behavioral features add interpretable, predictive value beyond interaction patterns.

### Business Value:
- **Recommendation System Foundation**: SVD learned factors can seed a real-time offer-recommendation engine: *"Customers similar to user X tend to succeed with offers similar to Y."*
- **Cold-Start Mitigation**: New customers/offers can be provisioned with factor estimates based on aggregated behavior, enabling recommendations without extensive personalization.
- **Campaign Segmentation**: Latent factors reveal which customer cohorts align with which offer archetypes, informing targeted campaign design.

In [38]:
# ── SVD Collaborative Filtering ─────────────────────────────────
# Matrix Factorization via SVD for latent user-offer preferences.
# This is a classic collaborative-filtering approach: we treat the binary
# target (e.g. offer response) as a "rating", learn low-rank latent factors
# that capture hidden user tastes and offer characteristics, then reconstruct
# a dense prediction matrix. It shines when similar users respond similarly
# to similar offers, even if the raw matrix is very sparse.
from scipy.sparse.linalg import svds

# 1. Build user-offer matrix (ONLY from train data – never leak validation!)
train_svd_df = df.iloc[train_idx][['person', 'offer_id', target_col]].copy()

user_offer_matrix = train_svd_df.pivot_table(
    index='person', columns='offer_id', values=target_col, fill_value=0
)

print(f"User-Offer Matrix: {user_offer_matrix.shape[0]:,} users × {user_offer_matrix.shape[1]:,} offers")
print(f"   → Total cells: {user_offer_matrix.size:,}")
print(f"   → Sparsity:    {(user_offer_matrix == 0).sum().sum() / user_offer_matrix.size:.1%}")
# High sparsity is normal and actually helpful for SVD – most users only see a few offers.

# 2. SVD decomposition (low-rank approximation with k latent factors)
# We deliberately keep k small (max 5) so the model learns the *main* patterns
# rather than memorizing noise. This is the "collaborative signal".
k = min(5, user_offer_matrix.shape[1] - 1)
U, sigma, Vt = svds(user_offer_matrix.values.astype(float), k=k)

# Reconstruct the approximated matrix (this fills in the missing values!)
# Clip to [0, 1] because our target is binary – the values can now be interpreted
# as predicted probability of positive response.
svd_pred_matrix = pd.DataFrame(
    (U @ np.diag(sigma) @ Vt).clip(0, 1),
    index=user_offer_matrix.index,
    columns=user_offer_matrix.columns,
)

print(f"   → SVD with k={k} latent factors completed")
print(f"      (singular values: {np.round(sigma, 3)} – larger values = more important patterns)")

# 3. Score validation set
valid_svd_df = df.iloc[valid_idx][['person', 'offer_id', target_col]].copy()
global_fallback = y_train.mean()   # overall positive rate in training data

svd_scores = valid_svd_df.apply(
    lambda row: (
        svd_pred_matrix.loc[row['person'], row['offer_id']]
        if row['person'] in svd_pred_matrix.index and row['offer_id'] in svd_pred_matrix.columns
        else global_fallback
    ),
    axis=1,
)

# Count how often we had to fall back (useful diagnostic)
num_fallback = (svd_scores == global_fallback).sum()
print(f"   → Global fallback (mean = {global_fallback:.1%}) used for "
      f"{num_fallback:,}/{len(valid_svd_df):,} validation rows "
      f"({num_fallback/len(valid_svd_df):.1%}) – these are cold-start users/offers.")

valid_pred_svd = (svd_scores >= 0.5).astype(int)

# 4. Evaluate
svd_metrics = {
    'model': 'svd_collaborative',
    'accuracy': accuracy_score(y_valid, valid_pred_svd),
    'balanced_accuracy': balanced_accuracy_score(y_valid, valid_pred_svd),
    'f1': f1_score(y_valid, valid_pred_svd),
    'precision': precision_score(y_valid, valid_pred_svd, zero_division=0),
    'recall': recall_score(y_valid, valid_pred_svd, zero_division=0),
    'roc_auc': roc_auc_score(y_valid, svd_scores),
    'pr_auc': average_precision_score(y_valid, svd_scores),
}

svd_mse = mean_squared_error(y_valid, svd_scores)

# Count fallback for better diagnosis
num_fallback = (svd_scores == global_fallback).sum()
fallback_pct = num_fallback / len(valid_svd_df) * 100

print(f"\nSVD Collaborative Filtering (k={k}) – Validation Results:")
print(f"   • Accuracy:           {svd_metrics['accuracy']:.4f}  (overall correct predictions)")
print(f"   • Balanced Accuracy:  {svd_metrics['balanced_accuracy']:.4f}  (handles class imbalance)")
print(f"   • Precision:          {svd_metrics['precision']:.4f}  (of predicted positives, how many were correct)")
print(f"   • Recall:             {svd_metrics['recall']:.4f}  (of actual positives, how many did we catch)")
print(f"   • F1-score:           {svd_metrics['f1']:.4f}  (harmonic mean of precision & recall)")
print(f"   • ROC-AUC:            {svd_metrics['roc_auc']:.4f}  (ranking quality)")
print(f"   • PR-AUC:             {svd_metrics['pr_auc']:.4f}  (most important for imbalanced data)")
print(f"   • MSE (on scores):    {svd_mse:.4f}  (how close predictions are to true 0/1 labels)")

print("\n=== Model Interpretation ===")
print("What does this SVD result really mean?")

# Fallback diagnosis
fallback_pct = num_fallback / len(valid_svd_df) * 100
if fallback_pct == 0:
    print("✅ Excellent: 0% global fallback.")
    print("   Every validation user had history in training → SVD could learn personalized patterns.")
else:
    print(f"   Global fallback used on {fallback_pct:.1f}% of validation rows.")

# Performance summary
print("\nSVD Performance vs Naïve Baseline (always predict 39.5%):")
print(f"   • PR-AUC : {svd_metrics['pr_auc']:.4f} vs baseline {baseline_pr_auc:.4f} "
      f"(+{svd_metrics['pr_auc'] - baseline_pr_auc:.4f})")
print(f"   • ROC-AUC: {svd_metrics['roc_auc']:.4f} (random = 0.5000)")

if svd_metrics['pr_auc'] > baseline_pr_auc + 0.01:
    print("   → SVD is capturing real latent user-offer preferences! Small but positive signal.")
else:
    print("   → Signal is still weak.")

# Threshold reality check
if svd_metrics['recall'] == 0 or svd_metrics['f1'] == 0:
    print("\n⚠️  At 0.5 threshold the model predicts ZERO positives.")
    print("   This is very common with pure SVD on sparse data (89.9% sparsity here).")
    print("   The raw svd_scores are still useful for **ranking** offers per user,")
    print("   but not yet reliable for hard classification at 0.5.")
    print("   Recommendation: Lower the threshold (e.g. 0.40–0.45) or use ranking metrics (Recall@K, NDCG).")
else:
    print("\n   The 0.5 threshold produces positive predictions → usable for classification too.")

# Final business takeaway
print("\nOverall takeaway for this project:")
print("   With only 8 offers total, pure SVD has limited power.")
print("   It adds mild collaborative signal on top of popularity, but tree-based models")
print("   (HistGradientBoosting, RandomForest) that use demographic + offer features")
print("   are likely to remain much stronger.")

User-Offer Matrix: 16,928 users × 8 offers
   → Total cells: 135,424
   → Sparsity:    89.9%
   → SVD with k=5 latent factors completed
      (singular values: [36.66  39.348 43.837 47.373 59.522] – larger values = more important patterns)
   → Global fallback (mean = 39.5%) used for 0/15,890 validation rows (0.0%) – these are cold-start users/offers.

SVD Collaborative Filtering (k=5) – Validation Results:
   • Accuracy:           0.6214  (overall correct predictions)
   • Balanced Accuracy:  0.5000  (handles class imbalance)
   • Precision:          0.0000  (of predicted positives, how many were correct)
   • Recall:             0.0000  (of actual positives, how many did we catch)
   • F1-score:           0.0000  (harmonic mean of precision & recall)
   • ROC-AUC:            0.5386  (ranking quality)
   • PR-AUC:             0.4039  (most important for imbalanced data)
   • MSE (on scores):    0.3559  (how close predictions are to true 0/1 labels)

=== Model Interpretation ===
What d

## Step 5c) Ranking Sanity Check: NDCG@K and Recall@K

This small check compares whether SVD can place each user's held-out next offer near the top of the 8-offer candidate list.
It matters here because the dataset is small and sparse, so SVD's main value is ranking quality and ensemble diversity, not raw classification dominance.

In [41]:
from sklearn.metrics import ndcg_score

offer_static_cols = [
    c for c in X_train.columns if c.startswith('channel_') or c.startswith('offer_type_') or c in [
        'reward', 'difficulty', 'duration', 'channel_count', 'reward_to_difficulty',
        'reward_per_day', 'difficulty_per_day', 'has_web_channel', 'has_email_channel',
        'has_mobile_channel', 'has_social_channel', 'net_value_score'
    ]
]
user_cols = [c for c in X_train.columns if c not in offer_static_cols]
offer_templates = df.groupby('offer_id')[offer_static_cols].first()
offer_order = offer_templates.index.tolist()
offer_index = {offer_id: idx for idx, offer_id in enumerate(offer_order)}
valid_meta = df.iloc[valid_idx][['person', 'offer_id']].reset_index(drop=True)

ranking_models = [best_model_name, 'svd_collaborative', 'most_popular_offer']
ranking_models = list(dict.fromkeys(ranking_models))

def build_candidate_frame(user_row):
    base = user_row[user_cols].to_dict()
    rows = []
    for offer_id in offer_order:
        row = base.copy()
        row.update(offer_templates.loc[offer_id].to_dict())
        rows.append(row)
    candidate_df = pd.DataFrame(rows, index=offer_order)
    return candidate_df.reindex(columns=X_train.columns, fill_value=0)

def score_candidates(model_name, user_id, candidate_df):
    if model_name == 'svd_collaborative':
        return svd_pred_matrix.loc[user_id, candidate_df.index].to_numpy()
    if model_name == 'most_popular_offer':
        return offer_pop_rate.reindex(candidate_df.index).fillna(y_train.mean()).to_numpy()
    return fitted_models[model_name].predict_proba(candidate_df)[:, 1]

def hit_at_k(y_true, y_score, k):
    return float(np.any(y_true[np.argsort(-y_score)[:k]] > 0))

k_values = [3, 5]
ranking_rows = []

for model_name in ranking_models:
    ndcg_totals = {k: 0.0 for k in k_values}
    recall_totals = {k: 0.0 for k in k_values}

    for row_idx, user_row in X_valid.reset_index(drop=True).iterrows():
        user_id = valid_meta.loc[row_idx, 'person']
        candidate_df = build_candidate_frame(user_row)
        y_true = np.zeros(len(offer_order))
        y_true[offer_index[valid_meta.loc[row_idx, 'offer_id']]] = 1
        y_score = score_candidates(model_name, user_id, candidate_df)

        for k in k_values:
            ndcg_totals[k] += ndcg_score([y_true], [y_score], k=k)
            recall_totals[k] += hit_at_k(y_true, y_score, k)

    ranking_rows.append({
        'model': model_name,
        **{f'ndcg@{k}': ndcg_totals[k] / len(X_valid) for k in k_values},
        **{f'recall@{k}': recall_totals[k] / len(X_valid) for k in k_values},
    })

ranking_df = pd.DataFrame(ranking_rows).sort_values('ndcg@3', ascending=False).reset_index(drop=True)
print('Ranking evaluation uses one held-out next offer per user across 8 candidate offers.')
display(ranking_df.round(4))

print("Interpretation:")
print("Ranking evaluation uses one held-out next offer per user across 8 candidate offers.")
print("\nKey takeaways:")
print("• HistGradientBoosting is the strongest ranker overall.")
print("• The simple 'most popular offer' baseline is surprisingly competitive.")
print("• SVD provides a mild collaborative signal (better than random) but is weaker than feature-rich models.")
print("\nRecommendation:")
print("• Use HistGradientBoosting (or ensemble) as the main model for both ranking and classification.")
print("• Keep SVD scores as an additional feature or ensemble member to capture user-offer affinity patterns.")
print("• For final deployment, focus on per-user ranking metrics (NDCG, Recall@K) rather than fixed 0.5 threshold.")

Ranking evaluation uses one held-out next offer per user across 8 candidate offers.


,model,ndcg@3,ndcg@5,recall@3,recall@5
0,hist_gb,0.2769,0.3715,0.3760,0.6115
1,most_popular_offer,0.2697,0.3707,0.3784,0.6255
2,svd_collaborative,0.1983,0.3057,0.2991,0.5589


Interpretation:
Ranking evaluation uses one held-out next offer per user across 8 candidate offers.

Key takeaways:
• HistGradientBoosting is the strongest ranker overall.
• The simple 'most popular offer' baseline is surprisingly competitive.
• SVD provides a mild collaborative signal (better than random) but is weaker than feature-rich models.

Recommendation:
• Use HistGradientBoosting (or ensemble) as the main model for both ranking and classification.
• Keep SVD scores as an additional feature or ensemble member to capture user-offer affinity patterns.
• For final deployment, focus on per-user ranking metrics (NDCG, Recall@K) rather than fixed 0.5 threshold.


## Step 6) Threshold Strategy for Campaign Activation

For campaign execution, threshold choice matters more than raw accuracy.
We sweep thresholds to balance precision and recall, then choose the best F1 threshold as a default operating point.

In [42]:
best_model_name = results_df.iloc[0]['model']
best_proba = proba_cache[best_model_name]

threshold_grid = np.linspace(0.10, 0.90, 33)
th_rows = []
for th in threshold_grid:
    p = (best_proba >= th).astype(int)
    th_rows.append({
        'threshold': round(float(th), 3),
        'precision': precision_score(y_valid, p, zero_division=0),
        'recall': recall_score(y_valid, p, zero_division=0),
        'f1': f1_score(y_valid, p, zero_division=0),
        'balanced_accuracy': balanced_accuracy_score(y_valid, p)
    })

th_df = pd.DataFrame(th_rows).sort_values('f1', ascending=False).reset_index(drop=True)
best_threshold = float(th_df.iloc[0]['threshold'])
best_pred = (best_proba >= best_threshold).astype(int)

print('Best model:', best_model_name)
print('Chosen threshold:', best_threshold)
print('Classification report at chosen threshold:')
print(classification_report(y_valid, best_pred, digits=4))
th_df.head(10)

Best model: hist_gb
Chosen threshold: 0.475
Classification report at chosen threshold:
              precision    recall  f1-score   support

           0     0.9036    0.8108    0.8547      9874
           1     0.7343    0.8580    0.7914      6016

    accuracy                         0.8287     15890
   macro avg     0.8189    0.8344    0.8230     15890
weighted avg     0.8395    0.8287    0.8307     15890



,threshold,precision,recall,f1,balanced_accuracy
0,0.475,0.734282,0.858045,0.791354,0.834431
1,0.425,0.715457,0.884807,0.791171,0.835203
2,0.450,0.724133,0.871343,0.790947,0.834547
3,0.400,0.706917,0.895279,0.790026,0.834565
4,0.375,0.699090,0.906749,0.789493,0.834476
5,0.500,0.741511,0.842088,0.788605,0.831617
6,0.350,0.690166,0.914561,0.786674,0.832205
7,0.325,0.682288,0.922041,0.784250,0.830223
8,0.525,0.748258,0.821144,0.783008,0.826411
9,0.300,0.675057,0.931682,0.782876,0.829220


## Step 7) Segment-Level Performance Check

Evaluate whether model quality is consistent across key business segments.
This helps detect overfitting to dominant groups and informs campaign fairness.

In [43]:
valid_view = X_valid.copy()
valid_view['y_true'] = y_valid.values
valid_view['y_pred'] = best_pred
valid_view['y_proba'] = best_proba

segment_cols = [c for c in ['age_group', 'income_group', 'offer_type'] if c in valid_view.columns]
seg_reports = {}
for col in segment_cols:
    seg_df = valid_view.groupby(col, observed=False).apply(
        lambda g: pd.Series({
            'records': len(g),
            'actual_rate': g['y_true'].mean(),
            'pred_rate': g['y_pred'].mean(),
            'precision': precision_score(g['y_true'], g['y_pred'], zero_division=0),
            'recall': recall_score(g['y_true'], g['y_pred'], zero_division=0)
        })
    ).reset_index()
    seg_df[['actual_rate', 'pred_rate', 'precision', 'recall']] = seg_df[['actual_rate', 'pred_rate', 'precision', 'recall']].round(4)
    seg_reports[col] = seg_df

for k, v in seg_reports.items():
    print('\nSegment report:', k)
    display(v.sort_values('records', ascending=False).head(15))


Segment report: age_group


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_23460\3590616328.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  seg_df = valid_view.groupby(col, observed=False).apply(
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_23460\3590616328.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  seg_df = valid_view.groupby(col, observed=False).apply(


,age_group,records,actual_rate,pred_rate,precision,recall
3,46-55,4948.0,0.3171,0.3569,0.7271,0.8184
5,66+,3733.0,0.4361,0.5189,0.7486,0.8907
4,56-65,3149.0,0.4278,0.5151,0.7312,0.8805
2,36-45,1851.0,0.4263,0.5019,0.7277,0.8568
1,26-35,1272.0,0.3200,0.3632,0.7229,0.8206
0,18-25,937.0,0.2946,0.3351,0.7389,0.8406



Segment report: income_group


,income_group,records,actual_rate,pred_rate,precision,recall
3,middle,6267.0,0.3546,0.4059,0.7323,0.8384
2,lower_middle,4273.0,0.3702,0.4154,0.7442,0.8350
4,upper_middle,2399.0,0.5085,0.6403,0.7350,0.9254
1,low,2003.0,0.2861,0.3375,0.6967,0.8220
0,high,948.0,0.4420,0.5264,0.7575,0.9021


## Step 8) Export Pre-Model Artifacts

Save data splits, model scores, chosen threshold, and feature governance so later modeling notebooks remain reproducible.

In [44]:
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'premodel'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_export = X_train.copy()
train_export[target_col] = y_train.values
valid_export = X_valid.copy()
valid_export[target_col] = y_valid.values
valid_export['pred_proba_best'] = best_proba
valid_export['pred_label_best'] = best_pred
valid_export['svd_score'] = svd_scores.values
valid_export['popular_score'] = valid_proba_popular


train_path = OUTPUT_DIR / 'train_split.csv'
valid_path = OUTPUT_DIR / 'valid_split_with_preds.csv'
scores_path = OUTPUT_DIR / 'baseline_model_scores.csv'
threshold_path = OUTPUT_DIR / 'threshold_sweep.csv'
config_path = OUTPUT_DIR / 'premodel_config.json'

train_export.to_csv(train_path, index=False)
valid_export.to_csv(valid_path, index=False)

# Thêm 2 baselines mới vào results
results.append(popular_metrics)     # từ Step 4b
results.append(svd_metrics)         # từ Step 5b

# Rebuild results_df với tất cả 5 models
results_df = pd.DataFrame(results).sort_values('pr_auc', ascending=False).reset_index(drop=True)

results_df.to_csv(scores_path, index=False)
th_df.to_csv(threshold_path, index=False)

config = {
    'target_col': target_col,
    'best_model_name': best_model_name,
    'best_threshold': best_threshold,
    'dropped_id_like_cols': id_like_cols,
    'dropped_leakage_cols': leakage_cols,
    'n_numeric_features': len(numeric_cols),
    'n_categorical_features': len(categorical_cols),
    'n_total_features': len(feature_cols),
    'svd_k': k,                                    # ← THÊM
    'models_compared': list(results_df['model']),   # ← THÊM
}

with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)

print('Saved artifacts:')
print('-', train_path)
print('-', valid_path)
print('-', scores_path)
print('-', threshold_path)
print('-', config_path)
results_df

Saved artifacts:
- d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_data_driven_marketing\starbucks_reward_data_marketing\outputs\premodel\train_split.csv
- d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_data_driven_marketing\starbucks_reward_data_marketing\outputs\premodel\valid_split_with_preds.csv
- d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_data_driven_marketing\starbucks_reward_data_marketing\outputs\premodel\baseline_model_scores.csv
- d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_data_driven_marketing\starbucks_reward_data_marketing\outputs\premodel\threshold_sweep.csv
- d:\Desktop_informations\SGK năm 3\SGK kì 2 năm 3\PowerBI_DataDriven_Mkt\Project_data_driven_marketing\starbucks_reward_data_marketing\outputs\premodel\premodel_config.json


,model,accuracy,balanced_accuracy,f1,precision,recall,roc_auc,pr_auc
0,hist_gb,0.829075,0.831617,0.788605,0.741511,0.842088,0.907268,0.821336
1,random_forest,0.800629,0.795869,0.746722,0.719347,0.776263,0.888374,0.788355
2,log_reg,0.789050,0.806393,0.759092,0.668650,0.877826,0.882908,0.769039
3,most_popular_offer,0.658087,0.604763,0.460316,0.571958,0.385140,0.668263,0.509985
4,most_popular_offer,0.658087,0.604763,0.460316,0.571958,0.385140,0.668263,0.509985
5,svd_collaborative,0.621397,0.500000,0.000000,0.000000,0.000000,0.538641,0.403897
6,svd_collaborative,0.621397,0.500000,0.000000,0.000000,0.000000,0.538666,0.403810
